In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv("../data/telco_churn_training.csv")

In [3]:
df.shape

(5000, 21)

In [4]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [5]:
df = df.fillna(0)
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
x = df.drop(columns=["customerID","Churn"])
y = df["Churn"]
print(x.shape)
print(y.shape)

(5000, 19)
(5000,)


In [7]:
x_trn , x_val , y_trn , y_val = train_test_split(
    x,
    y,
    test_size=0.05,
    random_state= 1,
    stratify=y

)




In [8]:
num = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
cat = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]
bin = [
    "SeniorCitizen"
]

In [9]:
preprocessor = ColumnTransformer([
    ("numbers",StandardScaler(),num),
    ("categorical",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat),
    ("binary","passthrough",bin)
])

In [10]:
X_train_processed = preprocessor.fit_transform(x_trn) #calculate some scalars to use it after that in transformation
X_val_processed  = preprocessor.transform(x_val)


In [12]:
print(X_train_processed.shape)
print(X_val_processed.shape)

(4750, 45)
(250, 45)


In [13]:
y_trn = y_trn.map({
    "No": 0,
    "Yes": 1
})

y_val = y_val.map({
    "No": 0,
    "Yes": 1
})

In [15]:
print(y_trn.unique())
print(y_trn.dtype)

[0 1]
int64


In [17]:
print(type(X_train_processed))
print(X_train_processed.dtype)

<class 'numpy.ndarray'>
float64


In [19]:
#Convert the data to Tensor and float(if needed)
#float for better acc from the neural networks + speed
x_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32
)
x_val_tensor = torch.tensor(
    X_val_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_trn.to_numpy(),
    dtype=torch.float32
)
y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
)

In [20]:
y_train_tensor = y_train_tensor.unsqueeze(1)
y_val_tensor = y_val_tensor.unsqueeze(1)


In [21]:
print(x_train_tensor.shape)
print(y_train_tensor.shape)

print(x_val_tensor.shape)
print(y_val_tensor.shape)

torch.Size([4750, 45])
torch.Size([4750, 1])
torch.Size([250, 45])
torch.Size([250, 1])


In [22]:
class model(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(45,100),
            nn.ReLU(),
            nn.Linear(100,1)
    )

    def forward(self,x):
        return self.network(x)


In [23]:
model_1 = model()
print(model_1)

model(
  (network): Sequential(
    (0): Linear(in_features=45, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=1, bias=True)
  )
)


In [25]:
loss_fn = nn.BCEWithLogitsLoss()

In [26]:
optimizer = torch.optim.Adam(
    model_1.parameters(),
    lr=0.05
)

In [27]:
train_dataset = TensorDataset(
    x_train_tensor,
    y_train_tensor
)

In [28]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
epochs = 20

train_losses = []
val_losses = []

for epoch in range(epochs):

    # Training
    model_1.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model_1(x_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # Validation 
    model_1.eval()
    with torch.no_grad():
        val_output = model_1(x_val_tensor)
        val_loss = loss_fn(val_output, y_val_tensor)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {val_loss.item():.4f}"
    )

Epoch 1/20 - Train Loss: 0.4935 - Val Loss: 0.5126
Epoch 2/20 - Train Loss: 0.4925 - Val Loss: 0.5337
Epoch 3/20 - Train Loss: 0.4954 - Val Loss: 0.5004
Epoch 4/20 - Train Loss: 0.4909 - Val Loss: 0.5216
Epoch 5/20 - Train Loss: 0.4987 - Val Loss: 0.5017
Epoch 6/20 - Train Loss: 0.4972 - Val Loss: 0.5051
Epoch 7/20 - Train Loss: 0.5026 - Val Loss: 0.5186
Epoch 8/20 - Train Loss: 0.4934 - Val Loss: 0.5165
Epoch 9/20 - Train Loss: 0.4926 - Val Loss: 0.5291
Epoch 10/20 - Train Loss: 0.4939 - Val Loss: 0.5064
Epoch 11/20 - Train Loss: 0.4871 - Val Loss: 0.5001
Epoch 12/20 - Train Loss: 0.4978 - Val Loss: 0.5190
Epoch 13/20 - Train Loss: 0.4934 - Val Loss: 0.5245
Epoch 14/20 - Train Loss: 0.4974 - Val Loss: 0.5221
Epoch 15/20 - Train Loss: 0.4985 - Val Loss: 0.5237
Epoch 16/20 - Train Loss: 0.4935 - Val Loss: 0.5301
Epoch 17/20 - Train Loss: 0.4948 - Val Loss: 0.5279
Epoch 18/20 - Train Loss: 0.4949 - Val Loss: 0.5444
Epoch 19/20 - Train Loss: 0.5038 - Val Loss: 0.5250
Epoch 20/20 - Train L